In [1]:
!git clone https://github.com/rimfa/shopee-product-matching.git

!git config --global user.email"arzun9530@gmail.com"
!git config --global user.name"rimfa"

Cloning into 'shopee-product-matching'...
remote: Enumerating objects: 16, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 16 (delta 1), reused 11 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (16/16), 1.65 MiB | 20.11 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [22]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q kagglehub

In [3]:
import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [4]:
root = kagglehub.competition_download("shopee-product-matching")

100%|██████████| 1.68G/1.68G [00:22<00:00, 80.2MB/s]

Extracting files...


In [5]:
print(root)

/root/.cache/kagglehub/competitions/shopee-product-matching


In [6]:
import pandas as pd
import torch
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import cv2
import math
import tensorflow as tf
import sys
import gc
import math
from tqdm import tqdm
import wandb
import os

In [7]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
device

device(type='cuda')

In [8]:
train_path = os.path.join(root, 'train.csv')
train_df = pd.read_csv(train_path)

train_images_path = os.path.join(root, 'train_images')
train_images = os.listdir(train_images_path)

In [9]:
train_df.head(2)

,posting_id,image,image_phash,title,label_group
0,train_129225211,0000a68812bc7e98c42888dfb1c07da0.jpg,94974f937d4c2433,Paper Bag Victoria Secret,249114794
1,train_3386243561,00039780dfc94d01db8676fe789ecd05.jpg,af3f9460c2838f0f,"Double Tape 3M VHB 12 mm x 4,5 m ORIGINAL / DO...",2937985045


In [10]:
%cd /content/shopee-product-matching

/content/shopee-product-matching


In [11]:
!mkdir src

In [12]:
%%writefile src/validator.py
import numpy as np
from sklearn.neighbors import NearestNeighbors

def calculate_f1(target, preds):
    target_matches = set(target.split())
    pred_matches = set(preds)

    common = target_matches & pred_matches
    if len(common) == 0:
        return 0.0

    precision = len(common) / len(pred_matches)
    recall = len(common) / len(target_matches)

    f1 = 2 * (precision * recall) / (precision + recall)
    return f1

def multimodal(query_df, query_hash, query_tfidf, gallery_hash, gallery_tfidf, gallery_df):
    res = []

    gallery_post_ids = gallery_df['posting_id'].values
    max_bits = 9

    hash_knn = NearestNeighbors(metric='hamming')
    hash_knn.fit(gallery_hash)
    hash_dist, hash_idxs = hash_knn.radius_neighbors(query_hash, radius=max_bits/64)

    tfidf_knn = NearestNeighbors(metric='cosine')
    tfidf_knn.fit(gallery_tfidf)
    tfidf_dist, tfidf_idxs = tfidf_knn.kneighbors(query_tfidf, n_neighbors=50)

    thresholds = np.arange(0.1, 0.4, 0.03)

    for bit in range(4, max_bits+1):
        for t in thresholds:
            f1_scores = []

            for i in range(len(query_df)):
                h_ind = hash_idxs[i][hash_dist[i] <= bit/64]
                img_neighb = gallery_post_ids[h_ind]

                tfidf_ind = tfidf_idxs[i][tfidf_dist[i] <= t]
                txt_neighb = gallery_post_ids[tfidf_ind]

                preds = set(txt_neighb) | set(img_neighb)
                f1_scores.append(calculate_f1(query_df['target_matches'].iloc[i], preds))

            mean_f1 = np.mean(f1_scores)
            res.append([bit, round(t, 4), round(mean_f1, 4)])

    return res


Writing src/validator.py


In [13]:
from torch.utils.data import Dataset, DataLoader
import cv2
import albumentations as A

class ShopeeImageDataset(Dataset):
  def __init__(self, root, transform, df):
    super().__init__()

    self.df = df
    self.root = root
    self.images_path = self.df['image'].values

    self.transform = transform

  def __len__(self):
    return len(self.images_path)

  def __getitem__(self, index):
    img_path = os.path.join(self.root, 'train_images', self.images_path[index])
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    if self.transform is not None:
      img = self.transform(image=img)['image']
    return img

In [14]:
from tqdm import tqdm
def get_embeddings(model, dataloader, device):

  embeddings = []
  model.eval()

  with torch.no_grad():
    for images in tqdm(dataloader, desc='getting embeddings'):
      images = images.to(device)
      emb = model(images).detach().cpu().numpy()

      embeddings.append(emb)

  embeddings = np.vstack(embeddings)
  return np.array(embeddings)

In [15]:
import timm

model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

In [16]:
from sklearn.model_selection import train_test_split
import re
# def hex_to_bits(phash):
#     binary = format(int(phash, 16), '064b')
#     return [int(bit) for bit in binary]

def clean_text(string):
    string = re.sub(r'[^а-яА-ЯёЁa-zA-Z0-9]', ' ', string)
    string = re.sub(r'\s+', ' ', string)
    return string.strip()

In [17]:
matches = train_df.groupby('label_group')['posting_id'].apply(lambda x: ' '.join(x))
train_df['target_matches'] = train_df['label_group'].map(matches)

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

#get clean title-corpus for each image
corpus = train_df['title'].apply(clean_text)
vectorizer = TfidfVectorizer(
    analyzer='word', ngram_range=(1, 2), max_features=27000
    )

#spit for getting groups(train and val)
group_id = train_df['label_group'].unique()
x_train_idx, x_val_idx = train_test_split(
    group_id, test_size=0.2, random_state=101
    )
train = train_df[train_df.label_group.isin(x_train_idx)] # [0, 3, 5 ....]
val = train_df[train_df.label_group.isin(x_val_idx)] #[1, 2, 4 .....]

train_tfidf = vectorizer.fit_transform(corpus[train.index]) # векторы для названий карточек
val_tfidf = vectorizer.transform(corpus[val.index]) # индексы все ще как в оригинальном массиве

train_df = train.reset_index(drop=True)
val_df = val.reset_index(drop=True)

In [19]:
from albumentations.pytorch import ToTensorV2

val_transforms = A.Compose([
    A.Resize(384, 384, p=1.0),
    A.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
        max_pixel_value=255.0,
        p=1.0
    ), ToTensorV2(p=1.0)
])

train_dataset = ShopeeImageDataset(root, val_transforms, train_df)
val_dataset = ShopeeImageDataset(root, val_transforms, val_df)

train_loader = DataLoader(dataset=train_dataset, batch_size=32, shuffle=False)
val_loader = DataLoader(dataset=val_dataset, batch_size=32, shuffle=False)

In [21]:
train_embeddings = get_embeddings(model, train_loader, device)
val_embeddings = get_embeddings(model, val_loader, device)

getting embeddings: 100%|██████████| 215/215 [01:07<00:00,  3.17it/s]


In [ ]:
from sklearn.neighbors import NearestNeighbors

def ImageRetrieval(train_emb, val_emb):

  emb_gallery = np.vstack((train_emb['embeddings'], val_emb['embeddings']))
  id_gallery = np.concatenate((train_emb['posting_ids'], val_emb['posting_ids']))

  emb_query = val_emb['embeddings']
  ind_query = val_emb['posting_ids']

  image_knn = NearestNeighbors(metric='cosine')
  image_knn.fit(emb_gallery)

  distances, indices = image_knn.radius_neighbors(emb_query, radius=0.4)

  return {'query_ids':ind_query,
          'gallery_ids': id_gallery,
          'indices': indices,
          'distances': distances
          }


In [25]:
import numpy as np
import os

path_to_emb = '/content/drive/MyDrive/shopee/artifacts/embeddings/'
os.makedirs(path_to_emb, exist_ok=True)

np.savez(os.path.join(path_to_emb, 'efficientnet_b0_train.npz'),
         embeddings=train_embeddings,
         posting_ids=train_df['posting_id'].values
)
np.savez(os.path.join(path_to_emb, 'efficientnet_b0_val.npz'),
         embeddings=val_embeddings,
         posting_ids=val_df['posting_id'].values
)
print(os.listdir(path_to_emb))

['efficientnet_b0_train.npz', 'efficientnet_b0_val.npz']


In [26]:
train_embs = np.load(os.path.join(path_to_emb, 'efficientnet_b0_train.npz'))
val_embs = np.load(os.path.join(path_to_emb, 'efficientnet_b0_val.npz'))

print(train_embs['embeddings'].shape)
print(val_embs['embeddings'].shape)

(27393, 1280)
(6857, 1280)


In [29]:
%cd /content/shopee-product-matching

/content/shopee-product-matching


In [31]:
!ls src
!ls notebooks

validator.py
baseline.ipynb


In [33]:
!ls "/content/drive/MyDrive/Colab Notebooks"

 01-ml-trainings-tabs.ipynb   EfficientNet-emb.ipynb	  image-eda.ipynb
'eda1 (1).ipynb'	      hw3_computer_vision.ipynb   S01-girafe-ai.ipynb
 eda1.ipynb		     'image-eda (1).ipynb'


In [ ]:
!cp "/content/drive/MyDrive/Colab Notebooks/EfficientNet-emb.ipynb" "/content/shopee-product-matching/notebooks/"